In [ ]:
import scanpy as sc
import os, random
import numpy as np

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

seed_value = 123

np.random.seed(seed_value)
random.seed(seed_value)


In [ ]:
def get_qc_thresholds(full_obs, sample_id, p_total_counts_q, p_n_genes_by_counts_q, p_pct_mt, p_bin_count_min, p_bin_count_max):
    filtered_obs = full_obs.obs[full_obs.obs["sample_id"] == sample_id].copy()
    total_counts_q = round(filtered_obs.total_counts.quantile(p_total_counts_q),0)
    n_genes_by_counts_q = round(filtered_obs.n_genes_by_counts.quantile(p_n_genes_by_counts_q),0)
    if total_counts_q == 0:
        total_counts_q = 1
    if n_genes_by_counts_q == 0:
        n_genes_by_counts_q = 1
    pct_mt = p_pct_mt
    bin_count_min = p_bin_count_min
    bin_count_max = p_bin_count_max
    if p_pct_mt is None:
        pct_mt = round(filtered_obs.pct_counts_mt.quantile(0.25),0)
    if p_bin_count_min is None:
        bin_count_min = round(filtered_obs.bin_count.quantile(0.05),0)
        if bin_count_min == 0:
            bin_count_min = 1
    if p_bin_count_max is None:
        bin_count_max = round(filtered_obs.bin_count.quantile(0.95),0)
    return [total_counts_q, n_genes_by_counts_q, pct_mt, bin_count_min, bin_count_max]

In [ ]:
adatas = []

run_name = "no_threshold_10x"
RAW_DIR = f"/fs/ess/PAS2713/aladyevae/work/2026_visium_hd_methodology/main_analysis/bin2cell/{run_name}"
samples_meta = {"AD2_S1": ["FF", "pilot_ff"],
                "AD2_S2": ["FF", "pilot_ff"],
                "AD3_S1": ["FF", "batch1_ff"],
                "AD4_S1": ["FF", "batch1_ff"],
                "AD5_S1": ["FF", "batch1_ff"],
                "ffpeAD2_S3": ["FFPE", "pilot_ffpe"],
                "ffpeAD2_S3": ["FFPE", "pilot_ffpe"]}

for sample_id in list(samples_meta.keys()):
    cdata = sc.read_h5ad(os.path.join(RAW_DIR, f"{sample_id}_b2c.h5ad"))
    cdata.X.data = np.round(cdata.X.data)
    cdata.raw = cdata.copy()
    # mitochondrial genes
    cdata.var["mt"] = cdata.var_names.str.startswith("MT-")
    # ribosomal genes
    cdata.var["ribo"] = cdata.var_names.str.startswith(("RPS", "RPL"))
    
    sc.pp.calculate_qc_metrics(
        cdata, qc_vars=["mt", "ribo"], inplace=True, log1p=True
    )
    
    cdata.obs['sample_id'] = sample_id
    cdata.obs['fixation_method'] = samples_meta[sample_id][0]
    cdata.obs['batch_id'] = samples_meta[sample_id][1]
    adatas.append(cdata)

In [ ]:
[x.shape[1] for x in adatas]

In [ ]:
merged_adatas = sc.concat(adatas)
merged_adatas.var_names_make_unique()
merged_adatas.obs_names_make_unique()
merged_adatas.layers["counts"] = merged_adatas.X.copy()
merged_adatas.shape

In [ ]:
merged_adatas.obs['sample_id'].value_counts()

In [ ]:
merged_adatas.write_h5ad("/fs/ess/PAS2713/aladyevae/work/2026_visium_hd_methodology/main_analysis/data/ffpe_ff_pilot_batch_1_pre_qc_v2.h5ad")

In [ ]:
qc_thresholds = {sample_id: get_qc_thresholds(merged_adatas, sample_id, 0.1, 0.1, np.float64(10.0), None, None) for sample_id in merged_adatas.obs["sample_id"].unique()}

In [ ]:
qc_thresholds

In [ ]:
adatas_filtered = []
for sample_id in merged_adatas.obs["sample_id"].unique():
    print(sample_id)
    adata = merged_adatas[merged_adatas.obs["sample_id"] == sample_id].copy()
    sc.pp.filter_cells(adata, min_counts = 1)
    sc.pp.filter_cells(adata, min_genes = 1)
    
    # mitochondrial genes
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    # ribosomal genes
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=["mt", "ribo"], inplace=True, log1p=True
    )
    print(adata.shape)
    
    qc_threshold = qc_thresholds[sample_id]
    adata_filtered = adata[
        (adata.obs['total_counts'] >= qc_threshold[0]) &
        (adata.obs['n_genes_by_counts'] >= qc_threshold[1]) &
        (adata.obs['pct_counts_mt'] < qc_threshold[2]) &
        (adata.obs['bin_count'] >= qc_threshold[3]) &
        (adata.obs['bin_count'] <= qc_threshold[4])].copy()
    print(adata_filtered.shape)
    adatas_filtered.append(adata_filtered)

In [ ]:
merged_filtered = sc.concat(adatas_filtered)
sc.pp.filter_genes(merged_filtered, min_cells=3)
# mitochondrial genes
merged_filtered.var["mt"] = merged_filtered.var_names.str.startswith("MT-")
# ribosomal genes
merged_filtered.var["ribo"] = merged_filtered.var_names.str.startswith(("RPS", "RPL"))

sc.pp.calculate_qc_metrics(
    merged_filtered, qc_vars=["mt", "ribo"], inplace=True, log1p=True
)
merged_filtered.var_names_make_unique()
merged_filtered.obs['source'] = "nuclei_after_QC"
print(merged_filtered.shape)

In [ ]:
merged_filtered.obs.sample_id.value_counts()

In [ ]:
### Add spatial context

# Initialize the spatial dict in the merged object
merged_filtered.uns['spatial'] = {}

# Re-map the spatial metadata back to the merged object
for i, sample_name in enumerate(list(samples_meta.keys())):
    # Retrieve the original spatial metadata for the specific sample
    merged_filtered.uns['spatial'][sample_name] = adatas[i].uns['spatial'][sample_name]

In [ ]:
merged_filtered.shape

In [ ]:
samples = merged_filtered.obs['sample_id'].unique()

for i, sample in enumerate(samples):
    subset = merged_filtered[merged_filtered.obs['sample_id'] == sample]
    
    sc.pl.spatial(
        subset, 
        library_id=sample, 
        color='log1p_n_genes_by_counts',
        title=f"Sample: {sample}",
        s=8, alpha_img=0
    )

In [ ]:
# Backup raw counts into a dedicated layer
merged_filtered.layers["counts"] = merged_filtered.X.copy()

merged_filtered.write_h5ad("/fs/ess/PAS2713/aladyevae/work/2026_visium_hd_methodology/main_analysis/data/ffpe_ff_pilot_batch_1_post_qc_v2.h5ad")